# Budget Curves — mAP vs AL Cycle
Reproduces Figure 3a from the paper: mAP after each AAS cycle and
the actual annotation budget utilised per cycle.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import glob
import os
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Load per-run results
# NOTE: Add per-cycle mAP logging to train_aas.py if you want cycle-level curves.
# Currently train_aas.py saves a single end-of-training JSON per run.
RESULTS_DIR = '../experiments/results'

results = []
for f in sorted(glob.glob(os.path.join(RESULTS_DIR, '*.json'))):
    if 'summary' in f:
        continue
    with open(f) as fh:
        results.append(json.load(fh))

print(f'Loaded {len(results)} result files')

In [ ]:
# Budget utilisation per run
budgets  = [r.get('budget_pct', 0) for r in results]
datasets = [r['dataset'] for r in results]
maps     = [r['metrics'].get('mAP', 0) * 100 for r in results]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# mAP distribution
axes[0].hist(maps, bins=20, edgecolor='black')
axes[0].axvline(56.14, color='red', linestyle='--', label='Paper target (56.14%)')
axes[0].set_xlabel('mAP (%)')
axes[0].set_ylabel('Count')
axes[0].set_title('mAP distribution across runs')
axes[0].legend()

# Budget utilisation
axes[1].hist(budgets, bins=20, edgecolor='black', color='orange')
axes[1].axvline(0.033, color='red', linestyle='--', label='Paper (0.033%)')
axes[1].set_xlabel('Budget used (%)')
axes[1].set_ylabel('Count')
axes[1].set_title('Annotation budget utilisation')
axes[1].legend()

plt.tight_layout()
plt.savefig('../experiments/results/budget_curves.png', dpi=150)
plt.show()

print(f'Mean mAP:    {np.mean(maps):.2f}%  (target: 56.14%)')
print(f'Mean budget: {np.mean(budgets):.4f}%  (target: 0.033%)')